In [ ]:
# Iris 鸢尾花分类项目

## 1. 导入库

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import os
os.makedirs('../results/figures', exist_ok=True)

## 2. 加载数据

iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"特征名称: {feature_names}")
print(f"类别名称: {target_names}")
print(f"数据形状: {X.shape}")
print(f"类别分布: {np.bincount(y)}")

## 3. 数据探索与可视化

df = pd.DataFrame(X, columns=feature_names)
df['species'] = y
df['species_name'] = df['species'].map({i: name for i, name in enumerate(target_names)})

df.head()
df.describe()
df.isnull().sum()

### 3.1 类别分布图
plt.figure(figsize=(6, 4))
sns.countplot(x='species_name', data=df, palette='viridis')
plt.title('Iris 类别分布')
plt.tight_layout()
plt.savefig('../results/figures/iris_distribution.png', dpi=150)
plt.show()

### 3.2 特征散点图
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='petal length (cm)', y='petal width (cm)', 
                hue='species_name', palette='deep', s=80)
plt.title('花瓣长 vs 花瓣宽（按类别着色）')
plt.tight_layout()
plt.savefig('../results/figures/iris_feature_plot.png', dpi=150)
plt.show()

### 3.3 相关性热力图
plt.figure(figsize=(8, 6))
sns.heatmap(df[feature_names].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Iris 特征相关性热力图')
plt.tight_layout()
plt.savefig('../results/figures/iris_correlation.png', dpi=150)
plt.show()

## 4. 模型训练

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}

results = {}
conf_matrices = {}

for name, model in models.items():
    if name in ['Logistic Regression', 'KNN (k=5)']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    results[name] = acc
    conf_matrices[name] = cm
    print(f"{name}: 准确率 = {acc:.4f}")
    print(f"混淆矩阵:\n{cm}\n")

## 5. 混淆矩阵可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, cm) in zip(axes, conf_matrices.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=target_names, yticklabels=target_names)
    ax.set_title(f'{name}\n准确率 = {results[name]:.4f}')
plt.tight_layout()
plt.savefig('../results/figures/iris_confusion_matrix.png', dpi=150)
plt.show()

## 6. 特征重要性分析
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X, y)
importances = dt.feature_importances_
for name, imp in zip(feature_names, importances):
    print(f"{name}: {imp:.4f}")

print(f"\n结论：最重要的特征是 '{feature_names[np.argmax(importances)]}'")